# CSR-Preprocessor:
File: fluxy/scripts/preprocessing/csr/csr_preprocess.ipynb

This notebook contains the full preprocessing chain, converting output from CarboScope Regional (CSR) model to fluxy-readable NetCDF file.

The preprocessor works as follows:

- Section 1. Create flux file for fluxy
- Section 2. Create concentration file for fluxy

Inputs:
- paths to CSR output (including prior and posterior files), according to the template below.
- selection of the species - please note that only `ch4` and `co2` are currently handled
- output paths to the processed files. The output will be a NetCDF (.nc) file, directory needs to be accessible. Please choose a descriptive filename and follow any naming conventions required by fluxy (see below). See fluxy documentation for appropriate filename convention: https://github.com/openghg/fluxy

Outputs:
- NetCDF flux output, fluxy format (e.g. `CSR_ch4_monthly.nc`)
- NetCDF concentration output, fluxy format (e.g. `CSR_ch4_monthly_concentrations.nc`)
- these can be analysed using fluxy tools, see `fluxy/scripts/mpi_bgc_itms_inversion_results.ipynb` for details

Note:
- csr_preprocessor.py and csr_preprocessor_concentration.py need to be present in `/scripts/preprocessing/csr/`
- output for `species = "co2"` is only handled by `fluxy/scripts/mpi_bgc_itms_inversion_results.ipynb` at the moment of writing

## 1. Create flux file for fluxy

In [ ]:
import sys
import os
from scripts.preprocessing.csr import csr_preprocessor as prp
import xarray as xr

# Add the path to the additional python files to system path
sys.path.append(os.path.abspath(".."))

##########
dirpath_csr_raw_output = "placeholder-path_to_csr_output"  # replace with actual path
# Example: dirpath_csr_raw_output = "/work/mj0143/b380914/INVERSION/OUTPUT/ch4_rn_inversion/2021_2023_inversion/ch4_only_test/OUTPUT14.019r8+stilt.2021-2023_stilt04.ch4.Version_27082025_MDMunc21_3ch4rnsigsel.dualtracer_v2025.ATMv2025_uba8_20250908_ADJrn-unc100ger/"
path_to_prior          = dirpath_csr_raw_output + "PRI_flux_monthly.nc"  # Path to CSR Prior-File
path_to_posterior      = dirpath_csr_raw_output + "mu1.0_100_flux_monthly.nc" # Path to CSR Posterior-File
##########
species = "ch4"  # species of the flux in the file. For example "ch4", "co2"
##########
dirpath_fluxy_output = "placeholder-path_to_fluxy_output"
# Example: dirpath_fluxy_output = "/work/bm1400/b380914/projects/fluxy-mpi-bgc/data/tests/CSR/"
path_to_output         = dirpath_fluxy_output + species + "/CSR_" + species + "_monthly.nc" # path to output
# Example: path_to_output = "/work/bm1400/b380914/projects/fluxy-mpi-bgc/data/tests/CSR/ch4/CSR_CH4_TEST_RHS_ch4_monthly.nc" # In order to be read by fluxy, the naming and storing conventions have to be followed (https://github.com/openghg/fluxy)

##########
path_to_prior_country = (
    dirpath_csr_raw_output + "PRI_flux_monthly_EUROPE30f.nc"
)  # other options: FLUXYALLf, FLUXYf
path_to_posterior_country = (
    dirpath_csr_raw_output + "mu1.0_100_flux_monthly_EUROPE30f.nc"
)
##########
# Provide paths to uncertainty files (RHS results, filenames: "cross.*")
path_to_uncertainty_country = ""
# Each flux time step (e.g., month, year) corresponds to one entry in a list "path_to_uncertainty_country",
# if no RHS results are available for a specific time step, enter empty path ("") for that entry.
# Example:
# path_to_uncertainty_country = ["/work/mj0143/b380914/INVERSION/OUTPUT/ch4_rn_inversion/2021_2023_inversion/ch4_only_test/rhs/OUTPUT14.019r8+stilt.2021-2023_stilt04.ch4.Version_27082025_MDMunc21_3ch4rnsigsel.dualtracer_v2025.ATMv2025_uba8_20250908_ADJrn_mu1._rhs=monthly+2021.04_EUROPE30f-unc100ger_2021jan/cross.rhs032_monthly+2021.04_EUROPE30f.d2",
#  "/work/mj0143/b380914/INVERSION/OUTPUT/ch4_rn_inversion/2021_2023_inversion/ch4_only_test/rhs/OUTPUT14.019r8+stilt.2021-2023_stilt04.ch4.Version_27082025_MDMunc21_3ch4rnsigsel.dualtracer_v2025.ATMv2025_uba8_20250908_ADJrn_mu1._rhs=monthly+2021.12_EUROPE30f-unc100ger_2021feb/cross.rhs032_monthly+2021.12_EUROPE30f.d2",
# ...
#  "/work/mj0143/b380914/INVERSION/OUTPUT/ch4_rn_inversion/2021_2023_inversion/ch4_only_test/rhs/OUTPUT14.019r8+stilt.2021-2023_stilt04.ch4.Version_27082025_MDMunc21_3ch4rnsigsel.dualtracer_v2025.ATMv2025_uba8_20250908_ADJrn_mu1._rhs=monthly+2023.96_EUROPE30f-unc100ger_2023dec/cross.rhs032_monthly+2023.96_EUROPE30f.d2" ]



# Remove old file if it exists
if os.path.exists(path_to_output):
    os.remove(path_to_output)

# Execute preprocessing script for fluxes
prp.preprocess(
    path_to_prior,
    path_to_posterior,
    path_to_prior_country,
    path_to_posterior_country,
    path_to_uncertainty_country,
    path_to_output,
    species,
)

## 2. Create concentration file for fluxy

In [ ]:
from scripts.preprocessing.csr import csr_preprocessor_concentration as prpc

##########
dirpath_csr_raw_output = "placeholder-path_to_csr_output"  # replace with actual path
# Example: dirpath_csr_raw_output = "/work/mj0143/b380914/INVERSION/OUTPUT/ch4_rn_inversion/2021_2023_inversion/ch4_only_test/OUTPUT14.019r8+stilt.2021-2023_stilt04.ch4.---.dualtracer_v2025_Val_20s.ATMv2025_Valentin_prior_ld_oc_nofrac_ADJrn/"
path_to_prior_conc     = dirpath_csr_raw_output + "PRI_output1.---.dualtracer_v2025_Val_20s/"
path_to_posterior_conc = dirpath_csr_raw_output + "mu1.0_100_output1.---.dualtracer_v2025_Val_20s/"
path_to_farfield_conc  = dirpath_csr_raw_output + "input.---.dualtracer_v2025_Val_20s/"
##########
species = "ch4"  # species of the flux in the file. For example "ch4", "co2"
##########
path_to_output_conc         = dirpath_fluxy_output + species + "/CSR_" + species + "_monthly_concentrations.nc" # path to output
# Example: path_to_output_conc    = "/work/bm1400/b380914/projects/fluxy-mpi-bgc/data/tests/CSR/ch4/CSR_CH4_TEST_RHS_ch4_monthly_concentrations.nc"
##########
start_year = 2021
end_year = 2023
##########

# Remove old file if it exists
if os.path.exists(path_to_output_conc):
    os.remove(path_to_output_conc)

# Execute preprocessing script for concentrations
prpc.preprocess_conc(
    path_to_prior_conc,
    path_to_posterior_conc,
    path_to_farfield_conc,
    path_to_output_conc,
    start_year,
    end_year,
    species,
)